In [17]:
import os
import yaml
import pandas as pd

metric_path = "../metric"
metric_list = []
# read all the .yaml files in the directory
for dir_name in os.listdir(metric_path):
    if dir_name.startswith(('csv', 'dl', 'human', 'bcb')) and os.path.isdir(os.path.join(metric_path, dir_name)):
        # read the .yaml file at dir_name/metric.yaml
        with open(os.path.join(metric_path, dir_name, 'metric.yaml'), 'r') as f:
            metric = yaml.safe_load(f)

        metric['name'] = dir_name
        metric_list.append(metric)


In [18]:
metric_list[0]

{'TMC-list': [{'code': 'def visual_quality(output, ground_truth):\n    from skimage.metrics import structural_similarity as ssim\n    output_image = output[0]  # This would require a method to convert the figure to an image array\n    ground_truth_image = ground_truth[0]  # Assuming the ground truth is also converted to an image array\n    return ssim(output_image, ground_truth_image, multichannel=True)',
   'metric': 'Visual Quality Assessment (e.g., using Structural Similarity Index)',
   'task_name': 'Data visualization'}],
 'prompt': "Draw normal distributions for multiple 'x' and 'y' arrays with labels. Each pair (x, y) represents a different chemical compound in the 'labels' list. The function should output with: tuple(fig: Matplotlib.figure.Figure)\nfig: Matplotlib figure object containing the drawn normal distributions.  \nYou should write self-contained code starting with:  \n```\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport scipy.stats as stats\ndef task_func(x

In [19]:
metric_name = dict()
task_name = dict()
for metric in metric_list:
    for tmc in metric['TMC-list']:
        if tmc['metric'] not in metric_name:
            metric_name[tmc['metric']] = {
                "counts": 1,
                "task_dict": {},
                "prompt_list": [],
                "task_id_list": []
            }
            metric_name[tmc['metric']]["task_dict"][tmc['task_name']] = 1
            metric_name[tmc['metric']]["prompt_list"].append(metric['prompt'])
            metric_name[tmc['metric']]["task_id_list"].append(metric['name'])
        else:
            metric_name[tmc['metric']]['counts'] += 1
            if tmc['task_name'] not in metric_name[tmc['metric']]["task_dict"]:
                metric_name[tmc['metric']]["task_dict"][tmc['task_name']] = 1
            else:
                metric_name[tmc['metric']]["task_dict"][tmc['task_name']] += 1
            metric_name[tmc['metric']]["prompt_list"].append(metric['prompt'])
            metric_name[tmc['metric']]["task_id_list"].append(metric['name'])
                
        if tmc['task_name'] not in task_name:
            task_name[tmc['task_name']] = 1
        else:
            task_name[tmc['task_name']] += 1

In [20]:
data = []

for metric, values in metric_name.items():
    for task, count in values["task_dict"].items():
        data.append({
            "Metric": metric,
            "Total Metric Count": values['counts'],  # total count for this metric
            "Task": task,
            "Task Count": count,  # count for this task under the specific metric
            "Prompts": values["prompt_list"],
            "Task ID": values["task_id_list"]
        })

# Convert the list of dictionaries to a DataFrame
df = pd.DataFrame(data)

In [21]:
df.head(3)

,Metric,Total Metric Count,Task,Task Count,Prompts,Task ID
0,"Visual Quality Assessment (e.g., using Structu...",1,Data visualization,1,[Draw normal distributions for multiple 'x' an...,[bcb659]
1,Covariance Matrix Shape,1,Data visualization,1,[Generates a pair plot from a numeric DataFram...,[bcb983]
2,Pair Plot Type,1,Data exploration and statistics understand,1,[Generates a pair plot from a numeric DataFram...,[bcb983]


In [22]:
# df = pd.DataFrame(list(metric_name.items()), columns=['Metric Name', 'Counts'])

# Display only the first 3 entries
df.head(3)

df.to_csv('metric_name_final.csv', index=False)

In [41]:
df = pd.DataFrame(list(task_name.items()), columns=['Task Name', 'Counts'])

# Display only the first 3 entries
df.head(3)

# Display the table
df.to_csv("task_name_counts.csv", index=False)